In [2]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import threading

In [2]:
def get_values(symbol):
    timezone = pytz.timezone("Etc/UTC")
    x = datetime.now()
    utc_from = datetime(2021, 6, 15, tzinfo=timezone)
    utc_to = datetime(x.year, x.month, x.day+1, tzinfo=timezone)
    rates = mt5.copy_rates_range(symbol, mt5.TIMEFRAME_M30, utc_from, utc_to)

    rates_frame = pd.DataFrame(rates)
    rates_frame = rates_frame.drop(['open', 'high', 'low','tick_volume', 'spread', 'real_volume'], axis=1)
    # convert time in seconds into the datetime format
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')

    EMA = rates_frame['close'].ewm(span=100, adjust=False).mean()
    DEMA = 2*EMA - EMA.ewm(span=100, adjust=False).mean()
    rates_frame['dma'] = DEMA
    rates_frame['ll'] = rates_frame['close'].rolling(window=100).mean()
    return rates_frame

In [3]:
#close order
def Action_close(ticket_no, symbol, signal):
    a = [[mt5.symbol_info_tick(symbol).bid, mt5.ORDER_TYPE_SELL], [mt5.symbol_info_tick(symbol).ask, mt5.ORDER_TYPE_BUY]]
    position_id=ticket_no
    price = a[signal][0]
    deviation=200
    request={
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": lot,
        "type": a[signal][1],
        "position": position_id,
        "price": price,
        "deviation": deviation,
        "magic": 234000,
        "comment": "python script close",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_RETURN,
    }
    result=mt5.order_send(request)
    dictt[symbol][signal] = 0
    print(result)


In [4]:
def Profit_checker(ticket_no, symbol, signal):
    print(ticket_no)
    while True:
        order_status = mt5.positions_get(ticket=ticket_no)
        profit = order_status[0].profit
        if profit >= 2.0:
            Action_close(ticket_no, symbol, signal)
            break
        elif profit <= -0.80:
            Action_close(ticket_no, symbol, signal)
            break
            #call close
        time.sleep(1)


In [5]:
def Action(symbol, lot, signal):
    symbol_info = mt5.symbol_info(symbol)
    if not symbol_info.visible:
        print(symbol, "is not visible, trying to switch on")
        if not mt5.symbol_select(symbol,True):
            print("symbol_select({}}) failed, exit",symbol)
            mt5.shutdown()
            quit()
    a = [[mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask], [mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid]]
    price = a[signal][1]
    deviation = 200
    
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": lot,
        "type": a[signal][0],
        "price": price,
        "deviation": deviation,
        "magic": 234000,
        "comment": "python script open",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_RETURN,
    }
    result = mt5.order_send(request)
    time.sleep(2)
    return result

In [ ]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit

In [ ]:
if not mt5.initialize():
    print("initialize() failed, error code =",mt5.last_error())
    quit()
global dictt
dictt = {'EURUSD':[0, 0], 'GBPUSD':[0, 0], 'USDJPY':[0, 0], 'EURGBP':[0, 0],'USDCHF':[0, 0], 'CADJPY':[0, 0]}
while True:
    for symbol in dictt:
        rates_frame = get_values(symbol)
        if rates_frame.iloc[-2].close > rates_frame.iloc[-1].dma and\
            rates_frame.iloc[-3].close < rates_frame.iloc[-1].dma \
            and dictt[symbol][0] == 0:
            lot = 0.01
            signal = 0
            
            ask=rates_frame.iloc[-2].dma
            bid=rates_frame.iloc[-1].close
            order_type = mt5.ORDER_TYPE_BUY
            buy_profit = price_action(symbol, lot, ask, bid, order_type)
            print(buy_profit)
            if buy_profit > -40:
                result = Action(symbol, lot, signal)
                p1 = threading.Thread(target=Profit_checker, args=(result.order, symbol, signal))
                p1.start()
                dictt[symbol][0] = 1
                dictt[symbol][1] = 0
#             result = Action(symbol, lot, signal)
#             time.sleep(3)
#             dictt[symbol][signal] = 1
#             dictt[symbol][1] = 0
#             p1 = threading.Thread(target=Profit_checker, args=(result.order, symbol, signal))
#             p1.start()
            
        elif rates_frame.iloc[-2].close < rates_frame.iloc[-1].dma and rates_frame.iloc[-3].close > rates_frame.iloc[-1].dma and dictt[symbol][1] == 0:
            lot = 0.01
            signal = 1
            ask=rates_frame.iloc[-2].dma
            bid=rates_frame.iloc[-1].close
            order_type = mt5.ORDER_TYPE_SELL
            buy_profit = price_action(symbol, lot, ask, bid, order_type)
            print(buy_profit)
            if buy_profit > -40: 
                result = Action(symbol, lot, signal)
                p1 = threading.Thread(target=Profit_checker, args=(result.order, symbol, signal))
                p1.start()
                dictt[symbol][1] = 1
                dictt[symbol][0] = 0
#             result = Action(symbol, lot, signal)
#             dictt[symbol][signal] = 1
#             dictt[symbol][0] = 0
#             time.sleep(3)
#             p1 = threading.Thread(target=Profit_checker, args=(result.order, symbol, signal))
#             p1.start()
            

        time.sleep(2)
    time.sleep(3)


In [3]:
import MetaTrader5 as mt5

mt5.initialize()
def Action(symbol, lot, signal):
    symbol_info = mt5.symbol_info(symbol)
    if not symbol_info.visible:
        print(symbol, "is not visible, trying to switch on")
        if not mt5.symbol_select(symbol,True):
            print("symbol_select({}}) failed, exit",symbol)
            mt5.shutdown()
            quit()
    a = [[mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask], [mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid]]
    price = a[signal][1]
    deviation = 200
    
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": lot,
        "type": a[signal][0],
        "price": price,
        "deviation": deviation,
        "magic": 234000,
        "comment": "python script open",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_RETURN,
    }
    result = mt5.order_send(request)
    time.sleep(2)
    return result
a = Action("EURUSD",0.01,0)

In [4]:
a

OrderSendResult(retcode=10004, deal=0, order=0, volume=0.0, price=0.0, bid=1.18426, ask=1.18433, comment='Requote', request_id=2, retcode_external=0, request=TradeRequest(action=1, magic=234000, order=0, symbol='EURUSD', volume=0.01, price=1.18455, stoplimit=0.0, sl=0.0, tp=0.0, deviation=200, type=0, type_filling=2, type_time=0, expiration=0, comment='python script open', position=0, position_by=0))